In [ ]:
# Final sanity checks and summary
print("="*70)
print("FINAL FEATURE ENGINEERING SUMMARY")
print("="*70)

# Row counts
print(f"\n📊 Data Volume:")
print(f"  Raw input rows:        {df_normalized.count():>10,}")
print(f"  After feature eng:     {df_features.count():>10,}")
print(f"  After null removal:    {df_clean.count():>10,}")

# Partition stats
print(f"\n⚙️  Spark Execution:")
print(f"  Input partitions:      {df_normalized.rdd.getNumPartitions():>10}")
print(f"  Feature partitions:    {df_features.rdd.getNumPartitions():>10}")
print(f"  Clean partitions:      {df_clean.rdd.getNumPartitions():>10}")

# Column coverage
print(f"\n✓ Features Generated ({len(df_clean.columns)} total columns):")
for col in sorted(df_clean.columns):
    print(f"  - {col}")

# Data quality checks
null_summary = df_clean.select(
    *[F.count(F.when(F.col(col).isNull(), col)).alias(col) for col in df_clean.columns]
).collect()[0]

print(f"\n🔍 Data Quality (null counts after cleaning):")
has_nulls = False
for col_name in df_clean.columns:
    null_count = getattr(null_summary, col_name)
    if null_count > 0:
        print(f"  {col_name}: {null_count}")
        has_nulls = True

if not has_nulls:
    print("  ✓ No nulls detected")

print(f"\n✓ Feature engineering EDA complete!")
print(f"✓ Output ready for BigQuery ML training")

# Don't stop spark in notebook — user may run more cells
print(f"\n💡 To run the full feature engineering job, execute:")
print(f"   python spark/feature_engineering.py --input <path> --output <path>")

## 12. Sanity Checks, Partition Stats, and Spark Shutdown

In [ ]:
# Assemble numeric features into a feature vector for correlation analysis
numeric_cols = [
    "units_sold", "revenue",
    "rolling_avg_units_7d", "rolling_avg_units_30d", "rolling_revenue_7d", "rolling_stddev_7d",
    "lag_1d_units", "lag_7d_units", "lag_14d_units", "lag_30d_units",
    "day_of_week", "week_of_year", "month", "is_weekend", "store_revenue_rank"
]

# Remove rows with any nulls in numeric features for correlation
df_numeric = df_clean.dropna(subset=numeric_cols)

# Assemble features
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vectors = assembler.transform(df_numeric)

# Compute Pearson correlation
pearson_corr_matrix = Correlation.corr(df_vectors, "features").collect()[0][0]

print("✓ Correlation matrix computed")
print(f"  Shape: {pearson_corr_matrix.numRows} x {pearson_corr_matrix.numCols}")
print(f"\n📊 Correlation Matrix (Top correlations with units_sold):")

# Extract correlation with units_sold (index 0)
units_sold_idx = 0
corr_with_units = []
for i, col_name in enumerate(numeric_cols):
    corr_val = pearson_corr_matrix[units_sold_idx, i]
    corr_with_units.append((col_name, corr_val))

# Sort by absolute correlation
corr_with_units.sort(key=lambda x: abs(x[1]), reverse=True)
for col_name, corr_val in corr_with_units[:10]:
    print(f"  {col_name:30s} : {corr_val:7.4f}")

## 11. Compute Correlation Matrix with pyspark.ml.stat.Correlation

In [ ]:
# 1. Global statistics using .describe()
print("📊 Descriptive Statistics (PySpark .describe()):")
df_clean.select(
    "units_sold", "revenue", "rolling_avg_units_7d", "rolling_stddev_7d",
    "lag_1d_units", "lag_7d_units", "store_revenue_rank"
).describe().show()

# 2. Distribution of units_sold per store
print("\n📊 Distribution of units_sold by store:")
(
    df_clean
    .groupBy("store_id")
    .agg(
        F.count("*").alias("days"),
        F.min("units_sold").alias("min"),
        F.percentile_approx("units_sold", 0.25).alias("q25"),
        F.percentile_approx("units_sold", 0.5).alias("median"),
        F.percentile_approx("units_sold", 0.75).alias("q75"),
        F.max("units_sold").alias("max"),
        F.avg("units_sold").alias("mean"),
        F.stddev("units_sold").alias("stddev")
    )
    .limit(10)
    .show()
)

# 3. Revenue aggregation by key dimensions
print("\n📊 Revenue by store_id (Top 10):")
(
    df_clean
    .groupBy("store_id")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(10)
    .show()
)

## 10. Run Scalable PySpark EDA Metrics

In [ ]:
# Null analysis before dropping
print("📊 Null counts before filtering:")
null_stats = df_features.select(
    *[F.count(F.when(F.col(col).isNull(), col)).alias(col) for col in df_features.columns]
).collect()[0]

for col_name in df_features.columns:
    null_count = getattr(null_stats, col_name)
    if null_count > 0:
        print(f"  {col_name}: {null_count}")

# Drop rows where lag features are null (cold-start period)
df_clean = df_features.dropna(subset=["lag_1d_units", "lag_7d_units"])

print(f"\n✓ Records after removing lag nulls: {df_clean.count():,}")
print(f"  (dropped cold-start period for lag_1d and lag_7d)")

# Show final schema
print(f"\n✓ Final feature table schema:")
df_clean.printSchema()

## 9. Handle Nulls and Prepare Output

In [ ]:
# Rank stores by daily revenue (competitive context signal)
rank_window = Window.partitionBy("date").orderBy(F.desc("revenue"))
df_features = df_features.withColumn("store_revenue_rank", F.rank().over(rank_window))

print("✓ Store revenue rank computed")
print(f"\n📊 Top 10 stores by revenue (single day sample):")
(
    df_features
    .where(F.col("date") == F.col("date").first())
    .select("date", "store_id", "revenue", "store_revenue_rank")
    .orderBy("store_revenue_rank")
    .limit(10)
    .show()
)

## 8. Compute Daily Store Revenue Rank

In [ ]:
# Extract calendar features
df_features = (
    df_features
    .withColumn("day_of_week", F.dayofweek("date"))  # 1=Sun, 7=Sat
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("month", F.month("date"))
    .withColumn("is_weekend", (F.dayofweek("date").isin(1, 7)).cast("int"))
)

print("✓ Calendar features added: day_of_week, week_of_year, month, is_weekend")
print(f"\n📊 Day-of-week distribution:")
(
    df_features
    .groupBy("day_of_week")
    .agg(
        F.count("*").alias("count"),
        F.avg("revenue").alias("avg_revenue"),
        F.avg("units_sold").alias("avg_units")
    )
    .orderBy("day_of_week")
    .show()
)

## 7. Add Calendar and Weekend Signals

In [ ]:
# Create lag features for time-series forecasting
for lag_days in [1, 7, 14, 30]:
    col_name = f"lag_{lag_days}d_units"
    df_features = df_features.withColumn(col_name, F.lag("units_sold", lag_days).over(lag_window))

print("✓ Lag features created: lag_1d_units, lag_7d_units, lag_14d_units, lag_30d_units")
print(f"\n📝 Sample with lags (date, store, units_sold, lag_1d, lag_7d):")
(
    df_features
    .select("date", "store_id", "units_sold", "lag_1d_units", "lag_7d_units")
    .where(F.col("lag_7d_units").isNotNull())
    .limit(5)
    .show()
)

## 6. Generate Lag Features (1d/7d/14d/30d)

In [ ]:
# Compute rolling aggregations
df_features = (
    df_normalized
    .withColumn("rolling_avg_units_7d", F.avg("units_sold").over(window_7d))
    .withColumn("rolling_avg_units_30d", F.avg("units_sold").over(window_30d))
    .withColumn("rolling_revenue_7d", F.sum("revenue").over(window_7d))
    .withColumn("rolling_stddev_7d", F.stddev("units_sold").over(window_7d))
)

print("✓ Rolling aggregations computed")
print(f"\n📊 Rolling features sample (date, store, rolling_avg_7d, rolling_stddev_7d):")
(
    df_features
    .select("date", "store_id", "rolling_avg_units_7d", "rolling_stddev_7d")
    .where(F.col("rolling_avg_units_7d").isNotNull())
    .limit(10)
    .show()
)

## 5. Compute Rolling Aggregation Features (7d/30d)

In [ ]:
# Define explicit window specs for feature engineering
# ────────────────────────────────────────────────────────────

# 7-day and 30-day rolling windows (range-based for calendar days)
window_7d = (
    Window.partitionBy("product_id", "store_id")
    .orderBy("date")
    .rangeBetween(-6 * 86400, 0)  # 6 days in seconds
)

window_30d = (
    Window.partitionBy("product_id", "store_id")
    .orderBy("date")
    .rangeBetween(-29 * 86400, 0)  # 29 days in seconds
)

# Lag window for time-series lags
lag_window = (
    Window.partitionBy("product_id", "store_id")
    .orderBy("date")
)

print("✓ Window specifications defined:")
print(f"  - window_7d:  partitionBy(product_id, store_id), rangeBetween(-6 days, 0)")
print(f"  - window_30d: partitionBy(product_id, store_id), rangeBetween(-29 days, 0)")
print(f"  - lag_window: partitionBy(product_id, store_id), orderBy(date)")

## 4. Define Time-Series Window Specifications

In [ ]:
# Normalize column names and cast types (adapting to Rossmann schema)
df_normalized = (
    df
    .withColumnRenamed("Store", "store_id")
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Sales", "revenue")  # Rossmann uses Sales instead of revenue
    .withColumn("store_id", F.col("store_id").cast("int"))
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("revenue", F.col("revenue").cast("float"))
)

# Rename Customers to units_sold (approximation for demo)
if "Customers" in df_normalized.columns:
    df_normalized = df_normalized.withColumnRenamed("Customers", "units_sold")
else:
    df_normalized = df_normalized.withColumn("units_sold", F.col("revenue") / 10.0)  # Synthetic for demo

# Add product_id if missing
if "product_id" not in df_normalized.columns:
    df_normalized = df_normalized.withColumn("product_id", F.lit("PRODUCT_001"))

print("✓ Schema normalized")
df_normalized.printSchema()

## 3. Apply Schema Normalization and Data Type Casting

In [ ]:
# For local development, read from sample data
# In production, this would read from gs://bucket/sales/raw/*.parquet
input_path = "data/sample_sales/rossmann_sample.csv"

# Try CSV first (local dev), fall back to parquet (GCS production)
try:
    df = spark.read.option("inferSchema", "true").csv(input_path, header=True)
    print(f"✓ Loaded CSV from {input_path}")
except:
    input_path = input_path.replace(".csv", "/*.parquet")
    df = spark.read.parquet(input_path)
    print(f"✓ Loaded Parquet from {input_path}")

# Schema inspection
print(f"\n📊 Raw Data Schema:")
df.printSchema()

# Sample rows
print(f"\n📝 Sample rows:")
df.limit(5).show()

# Aggregated stats — single action
(
    df
    .select(F.count("*").alias("total_rows"),
            F.countDistinct("Store").alias("num_stores"),
            F.countDistinct("Date").alias("num_dates"))
    .show()
)

## 2. Load Raw Sales Data from GCS or Parquet

In [ ]:
# Build Spark session with adaptive query execution
spark = (
    SparkSession.builder
    .appName("RetailFeatureEngineeringEDA")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")  # Tune for your data size
    .getOrCreate()
)

print(f"✓ SparkSession initialized: {spark.version}")
print(f"✓ Master: {spark.sparkContext.master()}")
print(f"✓ Partitions: {spark.sparkContext.defaultParallelism}")

## 1. Initialize Spark Session and Runtime Config

In [ ]:
import os
import sys
from datetime import datetime

# Add the current project root to path for imports
sys.path.insert(0, os.getcwd())

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

print("✓ Imports loaded")

# PySpark Feature Engineering EDA

**Goal:** Exploratory data analysis at scale using PySpark DataFrames to validate feature engineering pipeline before BigQuery ML training.

**Dataset:** Rossmann Store Sales — 1,115 stores, Aug–Sep 2015, full historical transactions.

This notebook:
1. Loads raw sales data from GCS/Parquet
2. Engineers time-series features (rolling windows, lags, calendar signals)
3. Validates schema, checks null distributions, and computes correlations
4. Demonstrates scalable PySpark EDA patterns (no pandas collect, efficient aggregations)